# LLM Validity Audit

Audits a small fixed sample of generated images with Gemini Flash through OpenRouter. This checks whether the image visibly satisfies the natural-language requirement. It is an external sanity check, not ground truth.

Set `OPENROUTER_API_KEY` in Colab Secrets before running the audit cell.

In [ ]:
from pathlib import Path
import importlib
import os
import shutil
import subprocess
import sys

REPO_URL = "https://github.com/casparbreloh/rbt4dnn-seminar.git"
COLAB_ROOT = Path("/content/rbt4dnn-seminar")
candidates = [Path.cwd(), *Path.cwd().parents]
ROOT = next((path for path in candidates if (path / "data/requirements.csv").exists()), None)
if ROOT is None:
    ROOT = COLAB_ROOT
    if (ROOT / ".git").exists():
        subprocess.run(["git", "-C", str(ROOT), "fetch", "origin", "main"], check=True)
        subprocess.run(["git", "-C", str(ROOT), "reset", "--hard", "origin/main"], check=True)
    else:
        if ROOT.exists():
            shutil.rmtree(ROOT)
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(ROOT)], check=True)

data_command = [
    sys.executable,
    str(ROOT / "scripts/fetch_data.py"),
    "--dataset",
    "mnist",
    "--dataset",
    "celeba-hq",
    "--dataset",
    "sgsm",
]
subprocess.run(data_command, cwd=ROOT, check=True)
subprocess.run([*data_command, "--verify"], cwd=ROOT, check=True)

try:
    userdata = importlib.import_module("google.colab.userdata")
    key = userdata.get("OPENROUTER_API_KEY")
    if key:
        os.environ["OPENROUTER_API_KEY"] = key
except Exception:
    pass

for module in ["shared", "llm_validity_audit"]:
    sys.modules.pop(module, None)

EXPERIMENT = ROOT / "experiments" / "llm-validity-audit"
sys.path.insert(0, str(ROOT / "src"))
sys.path.insert(0, str(EXPERIMENT))
print(ROOT)
commit = subprocess.check_output(
    ["git", "-C", str(ROOT), "log", "-1", "--oneline"], text=True
).strip()
print(commit)

In [ ]:
from llm_validity_audit import run_audit

paths = run_audit(ROOT, samples_per_requirement=2)
for path in paths:
    print(path)